# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{getattr(metadata, 'name', None)}: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below we list all record sets and their fields in the dataset, using their unique Croissant `@id`s.

This sets up which data sources are available for extraction.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (@id: {f.id}, type: {f.data_type})")
    print()

## 3. Data Extraction
Let's load data from each record set into a DataFrame for further analysis. We use the record set and field `@id`s as referenced above.

**Note:** The record set `@id`s can be found from the previous section. Adjust these as necessary for different record sets.

We will load all record sets found in the dataset.

In [ ]:
# Collect all record set @id(s)
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load all records from each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# Show columns (field @id's/names) for the first record set
if record_set_ids:
    print(f"\nColumns for record set {record_set_ids[0]}:")
    print(list(dataframes[record_set_ids[0]].columns))
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the data. Here we demonstrate common EDA steps: filtering, normalizing a numeric field, and grouping by a categorical field.

Modify field `@id` variables below depending on which record set you explore.

In [ ]:
# Choose record set and field @ids for demonstration
if len(record_set_ids) > 0:
    target_record_set_id = record_set_ids[0]  # Picking the first record set
    df = dataframes[target_record_set_id]
    
    # Try to automatically suggest a numeric field and a group field
    numeric_field_id = None
    group_field_id = None
    
    # Try to infer which fields are numeric, e.g. those containing 'count', 'number', 'mw', 'abundance', etc.
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
        # try guessing common field names
        if isinstance(df[col].iloc[0], (int, float)):
            numeric_field_id = col
            break
    # Try a group (categorical) field
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break
    
    print(f"Using record set: {target_record_set_id}")
    print(f"Numeric field candidate: {numeric_field_id}")
    print(f"Group field candidate: {group_field_id}\n")
    
    if numeric_field_id is not None:
        # Drop NaNs for analysis
        df_num = df[[numeric_field_id]].dropna()
        threshold = df_num[numeric_field_id].mean() if df_num[numeric_field_id].dtype in [int, float, 'float64'] else 10
        # Filtering
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}.")
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()])
        # Grouped analysis
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped by {group_field_id} (mean {numeric_field_id}):")
            display(grouped_df)
    else:
        print('No numeric field could be inferred; please check field names and types.')
else:
    print('No record sets found or loaded.')

## 5. Visualization
Visualize the numeric field and its distribution. If a group field exists, plot by group as well.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if a numeric field is available
if len(record_set_ids) > 0 and numeric_field_id:
    df = dataframes[target_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # If there is a group field, plot boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion

In this notebook, we successfully loaded and explored the [FAIR² Mass Spectrometry Human Mast Cell EVs dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the Croissant standard and `mlcroissant` API. We identified record sets, demonstrated typical exploratory analyses, and visualized key numeric variables.

**Key observations:**
- The dataset is structured for accurate FAIR data querying.
- Fields and their unique `@id`s provide transparent, reproducible references for programmatic workflows.
- Further biological/technical exploration may be tailored using the explicit Croissant schema mappings revealed here.

For additional analyses, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant) and the [FAIR² dataset schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).